<a href="https://colab.research.google.com/github/shiwangiedulearn-jpg/ai-engineer-learning/blob/main/vector_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q qdrant-client sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 6.3 MB/s eta 0:00:00


In [2]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

In [3]:
model= SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
print(model.get_sentence_embedding_dimension())#how many nos. in each embedding

384


/tmp/ipykernel_1288/82318877.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(model.get_sentence_embedding_dimension())#how many nos. in each embedding


In [6]:
documents = [
    "High blood pressure, also called hypertension, means that the force of blood against artery walls is consistently too high.",

    "Regular physical activity can support cardiovascular health and may help reduce cardiovascular risk.",

    "High cholesterol can contribute to plaque buildup in arteries and increase cardiovascular risk.",

    "Diabetes is a condition involving abnormal blood glucose regulation and can affect multiple organs when uncontrolled."
]

In [8]:
embeddings = model.encode(documents)
print(embeddings.shape)

(4, 384)


In [10]:
client = QdrantClient(":memory:")

In [14]:
collection_name= "medical_documents"
client.create_collection(
    collection_name= collection_name,
    vectors_config= models.VectorParams(
        size= model.get_sentence_embedding_dimension(),
        distance= models.Distance.COSINE
    )
)

/tmp/ipykernel_1288/1666538014.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  size= model.get_sentence_embedding_dimension(),


True

In [15]:
points =[]
for i, (document, embedding) in enumerate(zip(documents, embeddings)):
  points.append(
      models.PointStruct(
          id= i,
          vector= embedding.tolist(),
          payload={
              "text": document
          }
      )
  )

In [16]:
client.upsert(
    collection_name= collection_name,
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [27]:
query= "What happens when blood sugar is not controlled?"
query_embedding = model.encode(query).tolist()

In [28]:
results= client.query_points(
    collection_name= collection_name,
    query= query_embedding,
    limit=2,
    with_payload= True
).points

In [29]:
for result in results:
  print("Score:",round(result.score, 4))
  print("Text:", result.payload["text"])

Score: 0.6162
Text: Diabetes is a condition involving abnormal blood glucose regulation and can affect multiple organs when uncontrolled.
Score: 0.1957
Text: High blood pressure, also called hypertension, means that the force of blood against artery walls is consistently too high.
